## Setup

In [ ]:
#Download the needed packages.
import wrds
import os
import pandas as pd
from pathlib import Path

In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

In [ ]:
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

In [ ]:
db = wrds.Connection() #achieve connection
print("Connected")

## Get CRSP identifiers for SP500 companies.

In [ ]:
sp500 = db.raw_sql("""
SELECT 
    a.permno,
    b.permco,
    b.date,
    a.start,
    a.ending
FROM crsp.dsp500list AS a
JOIN crsp.dsf AS b 
ON a.permno = b.permno
AND b.date >= a.start
AND b.date <= COALESCE(a.ending, '9999-12-31')
WHERE b.date BETWEEN '2020-01-01' AND '2024-12-31'
ORDER BY b.date
""", date_cols=['start','ending','date'])

In [ ]:
sp500.sample(5)

In [ ]:
# Add Other Descriptive Variables

mse = db.raw_sql("""
                        select comnam, ncusip, namedt, nameendt, 
                        permno, ticker
                        from crsp.msenames
                        """, date_cols=['namedt', 'nameendt'])

# if nameendt is missing then set to today date
mse['nameendt']=mse['nameendt'].fillna(pd.to_datetime('today'))

In [ ]:
# Merge with SP500 data
sp500_full = pd.merge(sp500, mse, how = 'left', on = 'permno')

# Impose the date range restrictions
sp500_full = sp500_full.loc[(sp500_full.date>=sp500_full.namedt) \
                            & (sp500_full.date<=sp500_full.nameendt)]
sp500_full.sample(5)

## Get Compustat identifiers for SP500

In [ ]:
# Linking with Compustat through CCM

ccm= db.raw_sql("""
                  select gvkey, liid as iid, lpermno as permno, linktype, linkprim, 
                  linkdt, linkenddt
                  from crsp.ccmxpf_linktable
                  where substr(linktype,1,1)='L'
                  and (linkprim ='C' or linkprim='P')
                  """, date_cols=['linkdt', 'linkenddt'])

# if linkenddt is missing then set to today date
ccm['linkenddt']=ccm['linkenddt'].fillna(pd.to_datetime('today'))

### Merge CRSP and Compustat

In [ ]:
# Merge the CCM data with S&P500 data
# First just link by matching PERMNO
sp500ccm = pd.merge(sp500_full, ccm, how='left', on=['permno'])

# Then set link date bounds
sp500ccm = sp500ccm.loc[(sp500ccm['date']>=sp500ccm['linkdt'])\
                        &(sp500ccm['date']<=sp500ccm['linkenddt'])]
sp500ccm.sample(5)

In [ ]:
# Rearrange columns for final output

sp500ccm = sp500ccm.drop(columns=['namedt', 'nameendt', \
                                  'linktype', 'linkprim', 'linkdt', 'linkenddt'])

sp500ccm = sp500ccm[['date', 'permno', 'permco', 'comnam', 'ncusip', 'ticker', \
                     'gvkey', 'iid', 'start', 'ending']]
sp500ccm.sample(5)

In [ ]:
cnt = sp500ccm.groupby(['date'])['permno'].count().reset_index().rename(columns={'permno':'npermno'})


In [ ]:
db.describe_table('ciq', 'wrds_gvkey')

In [ ]:
ciq_link = db.raw_sql("""
SELECT companyid, gvkey, startdate, enddate
FROM ciq.wrds_gvkey
WHERE primaryflag = 1
""", date_cols=['startdate','enddate'])


In [ ]:
ciq_link['startdate'] = ciq_link['startdate'].fillna(pd.Timestamp('1900-01-01'))
ciq_link['enddate']   = ciq_link['enddate'].fillna(pd.Timestamp('today'))

In [ ]:
sp500_final = sp500ccm.merge(ciq_link, on='gvkey', how='left')

sp500_final = sp500_final[
    (sp500_final['date'] >= sp500_final['startdate']) &
    (sp500_final['date'] <= sp500_final['enddate'])
]

In [ ]:
sp500_final['gvkey_iid'] = sp500_final['gvkey'].astype(str) + '-' + sp500_final['iid'].astype(str)

In [ ]:
sp500_final = sp500_final.drop(columns=['startdate', 'enddate'])

In [ ]:
sp500_final

In [ ]:
daily_counts = (
    sp500_final.groupby('date')['permno']
    .nunique()
)

daily_counts.describe()

## Create link table

In [ ]:
link_table = sp500_final[
    ['companyid', 'gvkey', 'iid', 'gvkey_iid', 'permco', 'permno']
]

link_table = link_table.drop_duplicates()

In [ ]:
link_table.shape

## VALIDITY CHECKS

In [ ]:
sp500_final['companyid'].nunique()

In [ ]:
sp500_final['permco'].nunique()

In [ ]:
sp500_final['gvkey_iid'].nunique()

In [ ]:
sp500_final.groupby(['companyid','date'])['permno'].nunique().max()

In [ ]:
sp500_final.loc[sp500_final['companyid'] == 264164886.0]

In [ ]:
sp500_final.loc[sp500_final['gvkey_iid'] == '005073-19',
                ['comnam','ticker','gvkey','iid','permno','permco','companyid']].drop_duplicates()

In [ ]:
cid_multi = (
    sp500_final
    .groupby('companyid')['gvkey_iid']
    .nunique()
    .loc[lambda x: x > 1]
)

cid_multi

## WRTE TO CSV

In [ ]:
sp500_final.to_csv(DATA_RAW/"sp500_daily_membership.csv")

In [ ]:
link_table.to_csv(DATA_PROCESSED/'sp500_link_table.csv')

## CLOSE CONNECTION

In [ ]:
db.close()